# So sánh A/B: General mode vs IRAC mode

Eval trên **12 câu** (`test_subset_2mode.json`) phủ đủ 4 gap.  
Mỗi câu chạy **2 lần**: một lần `--response-mode general`, một lần `--response-mode irac`.  
Kết quả: F1 Khoản, F1 Điều, NormR per-question + aggregate.

> **Lưu ý**: cần Neo4j + Qdrant đang chạy và ANTHROPIC_API_KEY trong `.env`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root

from dotenv import load_dotenv
load_dotenv("../.env")

import json
import pandas as pd
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.3f}".format)

SUBSET_PATH = Path("../data/evaluation/test_subset_2mode.json")
OUT_DIR     = Path("../data/evaluation")
test_set    = json.loads(SUBSET_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(test_set)} câu từ {SUBSET_PATH.name}")
pd.DataFrame([{"id": q["id"], "gap": q["gap_type"], "question": q["question"][:70]} for q in test_set])

In [ ]:
# ---------------------------------------------------------------------------
# Khởi tạo shared clients (load 1 lần, dùng cho cả 2 mode)
# ---------------------------------------------------------------------------
from src.evaluation.run_evaluation import _build_shared_clients
print("Đang load Neo4j + Qdrant + BGE-M3 (30-60s lần đầu)...")
clients = _build_shared_clients()
print("✅ Clients ready")

In [ ]:
# ---------------------------------------------------------------------------
# Hàm helper: chạy 1 mode, trả về DataFrame per-question
# ---------------------------------------------------------------------------
from src.evaluation.run_evaluation import run_system_on_test_set

LLM_CACHE = OUT_DIR / ".llm_cache"

def run_mode(mode: str) -> pd.DataFrame:
    """Chạy graphrag với mode chỉ định, trả về DataFrame per-question."""
    print(f"\n{'='*60}")
    print(f"  Chạy mode: {mode.upper()}  ({len(test_set)} câu)")
    print(f"{'='*60}")
    results = run_system_on_test_set(
        test_set,
        system="graphrag",
        clients=clients,
        llm_cache_dir=LLM_CACHE,
        response_mode=mode,
    )
    rows = []
    for r in results:
        rows.append({
            "id":       r["id"],
            "gap":      r["gap_type"],
            "question": r["question"][:55],
            "F1_kh":    round(r["citation_score"]["f1"], 3),
            "F1_di":    round(r["citation_score_dieu"]["f1"], 3),
            "NormR":    round(r["norm_recall"], 3),
            "#pred":    len(r["pred_citations"]),
            "#gt":      len(r["ground_truth_citations"]),
            "neg_ok":   r["negative_correct"],
            "elapsed":  r["elapsed_seconds"],
            "mode":     r.get("response_mode", mode),
        })
    df = pd.DataFrame(rows)
    # Lưu raw results JSON để dùng compare_runs nếu cần
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    out = OUT_DIR / f"results_graphrag_{mode}_{ts}.json"
    out.write_text(json.dumps(
        {"system": "graphrag", "mode": mode, "test_set": str(SUBSET_PATH),
         "timestamp": ts, "results": results},
        ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  → Saved: {out.name}")
    return df

In [ ]:
# ---------------------------------------------------------------------------
# Chạy GENERAL mode
# ---------------------------------------------------------------------------
df_general = run_mode("general")
df_general

In [ ]:
# ---------------------------------------------------------------------------
# Chạy IRAC mode
# ---------------------------------------------------------------------------
df_irac = run_mode("irac")
df_irac

In [ ]:
# ---------------------------------------------------------------------------
# So sánh per-question: Δ F1_kh = irac - general
# ---------------------------------------------------------------------------
df_cmp = df_general[["id","gap","question","F1_kh","F1_di","NormR","#pred"]].copy()
df_cmp.columns = ["id","gap","question","G_F1kh","G_F1di","G_NormR","G_pred"]

df_cmp["I_F1kh"]  = df_irac["F1_kh"].values
df_cmp["I_F1di"]  = df_irac["F1_di"].values
df_cmp["I_NormR"] = df_irac["NormR"].values
df_cmp["I_pred"]  = df_irac["#pred"].values
df_cmp["ΔF1kh"]   = (df_cmp["I_F1kh"] - df_cmp["G_F1kh"]).round(3)

def color_delta(val):
    if val > 0.05:  return "background-color: #c6efce"   # green
    if val < -0.05: return "background-color: #ffc7ce"   # red
    return ""

print("Per-question: G = General | I = IRAC | Δ = I − G")
df_cmp.style.applymap(color_delta, subset=["ΔF1kh"])

In [ ]:
# ---------------------------------------------------------------------------
# Aggregate: mean F1 theo mode × gap
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

def agg(df, mode_label):
    rows = []
    # Overall (loại negative)
    sub = df[df["gap"] != "negative"]
    rows.append({"mode": mode_label, "gap": "OVERALL (non-neg)",
                 "F1_kh": sub["F1_kh"].mean(), "F1_di": sub["F1_di"].mean(),
                 "NormR": sub["NormR"].mean(), "n": len(sub)})
    # Per gap
    for gap in ["gap1","gap2","gap3","gap4"]:
        sub = df[df["gap"] == gap]
        if len(sub):
            rows.append({"mode": mode_label, "gap": gap,
                         "F1_kh": sub["F1_kh"].mean(), "F1_di": sub["F1_di"].mean(),
                         "NormR": sub["NormR"].mean(), "n": len(sub)})
    # Negative
    sub = df[df["gap"] == "negative"]
    rows.append({"mode": mode_label, "gap": "negative",
                 "F1_kh": None, "F1_di": None, "NormR": None,
                 "neg_ok": df_general[df_general["gap"]=="negative"]["neg_ok"].tolist(), "n": len(sub)})
    return pd.DataFrame(rows)

agg_df = pd.concat([agg(df_general, "general"), agg(df_irac, "irac")])
print("Aggregate F1 theo mode × gap:")
agg_df.set_index(["mode","gap"])[["F1_kh","F1_di","NormR","n"]]

In [ ]:
# ---------------------------------------------------------------------------
# Tóm tắt cuối: win/loss/tie giữa 2 mode
# ---------------------------------------------------------------------------
delta = df_cmp["ΔF1kh"]
wins   = (delta > 0.05).sum()
losses = (delta < -0.05).sum()
ties   = len(delta) - wins - losses

g_mean = df_general[df_general["gap"]!="negative"]["F1_kh"].mean()
i_mean = df_irac[df_irac["gap"]!="negative"]["F1_kh"].mean()

print(f"""\n{'='*55}
 TỔNG KẾT A/B  (12 câu / 4 gap)
{'='*55}
 General F1 Khoản (mean): {g_mean:.3f}
 IRAC    F1 Khoản (mean): {i_mean:.3f}
 Δ (IRAC − General):      {i_mean - g_mean:+.3f}

 Per-question  (|Δ| > 0.05):
   IRAC wins : {wins}
   IRAC losses: {losses}
   Ties       : {ties}
{'='*55}""")

# Top differences
print("\nCâu thay đổi nhiều nhất (|Δ| > 0.1):")
df_cmp[abs(df_cmp["ΔF1kh"]) > 0.1][["id","gap","question","G_F1kh","I_F1kh","ΔF1kh"]]